In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/DateFruit_Dataset (1).csv")

In [3]:
df.head() #34 features, 1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [5]:
df.shape


(898, 35)

In [8]:
x = df.drop("Class", axis = 1)
y = df["Class"]

In [9]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [10]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [14]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size = 0.2, random_state = 42
)

In [20]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [21]:
### ANN

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [25]:
x_train_tensor = torch.tensor(x_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long) #in classification, cross entropy function in used it expects target in long

x_test_tensor = torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [26]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

In [28]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [33]:
#Build the model

class ANN(nn.Module):
  def __init__(self):
    super(ANN, self).__init__()

    self.model = nn.Sequential(
        nn.Linear(x.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 7)
    )

  def forward(self, x):
    return self.model(x)

In [34]:
model = ANN()

#lOSS & optimizer
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [39]:
# Training the ANN
epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)   # Make sure it's 'criterion', not 'criteria'
        loss.backward()
        optimizer.step()  # Parameter update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}")

Epoch 1/100, Train Loss: 1.6420
Epoch 2/100, Train Loss: 1.0798
Epoch 3/100, Train Loss: 0.7031
Epoch 4/100, Train Loss: 0.5264
Epoch 5/100, Train Loss: 0.4357
Epoch 6/100, Train Loss: 0.3828
Epoch 7/100, Train Loss: 0.3403
Epoch 8/100, Train Loss: 0.3229
Epoch 9/100, Train Loss: 0.2954
Epoch 10/100, Train Loss: 0.2584
Epoch 11/100, Train Loss: 0.2457
Epoch 12/100, Train Loss: 0.2284
Epoch 13/100, Train Loss: 0.2183
Epoch 14/100, Train Loss: 0.2107
Epoch 15/100, Train Loss: 0.2076
Epoch 16/100, Train Loss: 0.1892
Epoch 17/100, Train Loss: 0.1886
Epoch 18/100, Train Loss: 0.1801
Epoch 19/100, Train Loss: 0.1747
Epoch 20/100, Train Loss: 0.1642
Epoch 21/100, Train Loss: 0.1646
Epoch 22/100, Train Loss: 0.1596
Epoch 23/100, Train Loss: 0.1528
Epoch 24/100, Train Loss: 0.1435
Epoch 25/100, Train Loss: 0.1359
Epoch 26/100, Train Loss: 0.1371
Epoch 27/100, Train Loss: 0.1328
Epoch 28/100, Train Loss: 0.1300
Epoch 29/100, Train Loss: 0.1321
Epoch 30/100, Train Loss: 0.1204
Epoch 31/100, Train

In [46]:
#Evalutate
model.eval()

total = 0
correct = 0

with torch.no_grad():
  for xb, yb in test_loader:
    outputs = model(xb) #[0.2, 0.5, 1.3, -0.5, , ,] -7 values
    _, predicted = torch.max(outputs, 1)

    correct += (predicted == yb).sum().item()
    total += yb.size(0) #actual samples in every batch

print("Accuracy: ",correct/total *100)

Accuracy:  96.66666666666667


In [49]:
y_test.shape

(180,)

In [ ]:
_